In [1]:
import sys
sys.path.append("/hlabhome/wg-mjames/git/Toby_work")
import pandas as pd
import numpy as np
import elephant
from scipy import stats,signal,integrate
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn import metrics, preprocessing
from sklearn.feature_selection import RFE
import imblearn
from imblearn import over_sampling
from imblearn.under_sampling import RandomUnderSampler
from scipy.stats import ks_2samp
import pickle
import random
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from graphical_functions import *
from helper_functions_math_addons import *
from helper_functions_file_handling import *
from helper_functions_modeling import *
from sklearn.utils.class_weight import compute_sample_weight
from collections import Counter
from threadpoolctl import threadpool_info, threadpool_limits
%load_ext autoreload
%autoreload 2
import os
NCPU = "4"
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family':'DejaVu Sans'})

In [2]:
split_thres = (0.4, 0.6)
addon = 'unfiltered'
downlims = [5,3,5]
load_data=True
train_scaled, test_scaled = obtain_train_test_scaled_data(split_thres, addon, load_data, 
                                                          downlims)

Loaded successfully


In [3]:
reversal_to_full = pd.concat([train_scaled, test_scaled])
condensed_dataframe = get_extended_condensed_data(reversal_to_full)

In [4]:
condensed_dataframe

,index,state,region,cell,animal,celltype,geno,cell_no,age,age_group,state_str,cell_state,state_chunks
0,0,2,ACad,caf69_358_1228_0_12_0_wt,caf69,RSU,wt,0,358,old,2,caf69_358_1228_0_12_0_wt-2,537
1,367,0,ACad,caf69_358_1228_0_12_0_wt,caf69,RSU,wt,0,358,old,0,caf69_358_1228_0_12_0_wt-0,199
2,524,1,ACad,caf69_358_1228_0_12_0_wt,caf69,RSU,wt,0,358,old,1,caf69_358_1228_0_12_0_wt-1,55
3,561,2,ACad,caf69_358_1228_0_12_1_wt,caf69,RSU,wt,1,358,old,2,caf69_358_1228_0_12_1_wt-2,94
4,627,0,ACad,caf69_358_1228_0_12_1_wt,caf69,RSU,wt,1,358,old,0,caf69_358_1228_0_12_1_wt-0,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12945,1824735,2,V1,bth100_99_0624_12_24_94_wt,bth100,RSU,wt,94,99,young,2,bth100_99_0624_12_24_94_wt-2,1241
12946,1825567,0,V1,bth100_99_0624_12_24_94_wt,bth100,RSU,wt,94,99,young,0,bth100_99_0624_12_24_94_wt-0,216
12947,1825748,1,V1,bth100_99_0624_12_24_94_wt,bth100,RSU,wt,94,99,young,1,bth100_99_0624_12_24_94_wt-1,33
12948,408415,1,ACad,zbr119_112_0604_0_12_75_app_ps1,zbr119,RSU,app_ps1,75,112,young,1,zbr119_112_0604_0_12_75_app_ps1-1,1


In [5]:
animals = condensed_dataframe.animal.unique()
output = {'animal':[], 'geno':[], 'age':[], 'age_group':[], 'sum_chunks':[],'chunks':[], 'unit_count':[]}
for animal in animals:
    data = condensed_dataframe[condensed_dataframe.animal==animal]
    output['animal'].append(animal)
    output['geno'].append(data.iloc[0].geno)
    output['age'].append(data.iloc[0].age)
    output['age_group'].append(data.iloc[0].age_group)
    output['unit_count'].append(len(data.drop(columns=['state', 'state_str', 'cell_state', 
                                                       'state_chunks','index']).drop_duplicates()))
    chunks = [[],[],[]]
    sum_chunks = [0,0,0]
    for i, row in data.iterrows():
        chunks[row.state].append(row.state_chunks)
        sum_chunks[row.state] += row.state_chunks
    output['chunks'].append(chunks)
    output['sum_chunks'].append(sum_chunks)
df = pd.DataFrame(output)

In [16]:
def permutations(*groups):
    result = []
    if len(groups)==1:
        return [[i] for i in groups[0]]
    base_groups = permutations(*groups[1:])
    for i, val in enumerate(groups[0]):
        for ind, li in enumerate(base_groups):
            addon = [val, *li]
            result.append(addon)
    return result

In [39]:
df

,animal,geno,age,age_group,sum_chunks,chunks,unit_count
0,caf69,wt,358,old,"[108444, 21244, 104313]","[[199, 59, 63, 65, 62, 78, 61, 126, 199, 83, 8...",366
1,caf71,app_ps1,492,old,"[60581, 1659, 35141]","[[122, 181, 205, 195, 179, 328, 198, 131, 166,...",248
2,caf73,app_ps1,381,old,"[152900, 8475, 50362]","[[113, 205, 794, 478, 635, 674, 307, 281, 212,...",389
3,caf74,app_ps1,419,old,"[97297, 1925, 60293]","[[399, 251, 126, 147, 314, 383, 813, 388, 250,...",289
4,caf75,app_ps1,389,old,"[67096, 3068, 57185]","[[815, 306, 162, 726, 1769, 277, 85, 252, 67, ...",309
5,caf77,wt,394,old,"[32964, 6729, 30492]","[[441, 463, 93, 649, 377, 101, 276, 178, 212, ...",112
6,caf81,wt,455,old,"[92061, 25413, 105843]","[[211, 150, 90, 171, 397, 75, 87, 270, 119, 15...",423
7,caf82,wt,455,old,"[81495, 14513, 66498]","[[1585, 635, 351, 144, 157, 777, 925, 864, 142...",210
8,zbr106,wt,101,young,"[55262, 12533, 94664]","[[119, 613, 284, 68, 177, 193, 1201, 88, 363, ...",242
9,zbr113,app_ps1,91,young,"[3653, 757, 21263]","[[60, 77, 105, 32, 208, 21, 18, 151, 159, 168,...",38


In [23]:
animal_ordered = []
for animal in df[(df.geno=='wt')&(df.age_group=='young')].animal.unique():
    animal_ordered.append(animal)
for animal in df[(df.geno=='app_ps1')&(df.age_group=='young')].animal.unique():
    animal_ordered.append(animal)
for animal in df[(df.geno=='wt')&(df.age_group=='old')].animal.unique():
    animal_ordered.append(animal)
for animal in df[(df.geno=='app_ps1')&(df.age_group=='old')].animal.unique():
    animal_ordered.append(animal)

In [25]:
animal_ordered.remove('zbr113')

In [26]:
animal_ordered

['zbr106',
 'bth90',
 'bth95',
 'bth100',
 'zbr114',
 'zbr117',
 'zbr118',
 'zbr119',
 'caf69',
 'caf77',
 'caf81',
 'caf82',
 'caf71',
 'caf73',
 'caf74',
 'caf75']

In [27]:
init_permut = permutations([0,1,2,3],[4,5,6,7],[8,9,10,11],[12,13,14,15])

In [28]:
init_permut

[[0, 4, 8, 12],
 [0, 4, 8, 13],
 [0, 4, 8, 14],
 [0, 4, 8, 15],
 [0, 4, 9, 12],
 [0, 4, 9, 13],
 [0, 4, 9, 14],
 [0, 4, 9, 15],
 [0, 4, 10, 12],
 [0, 4, 10, 13],
 [0, 4, 10, 14],
 [0, 4, 10, 15],
 [0, 4, 11, 12],
 [0, 4, 11, 13],
 [0, 4, 11, 14],
 [0, 4, 11, 15],
 [0, 5, 8, 12],
 [0, 5, 8, 13],
 [0, 5, 8, 14],
 [0, 5, 8, 15],
 [0, 5, 9, 12],
 [0, 5, 9, 13],
 [0, 5, 9, 14],
 [0, 5, 9, 15],
 [0, 5, 10, 12],
 [0, 5, 10, 13],
 [0, 5, 10, 14],
 [0, 5, 10, 15],
 [0, 5, 11, 12],
 [0, 5, 11, 13],
 [0, 5, 11, 14],
 [0, 5, 11, 15],
 [0, 6, 8, 12],
 [0, 6, 8, 13],
 [0, 6, 8, 14],
 [0, 6, 8, 15],
 [0, 6, 9, 12],
 [0, 6, 9, 13],
 [0, 6, 9, 14],
 [0, 6, 9, 15],
 [0, 6, 10, 12],
 [0, 6, 10, 13],
 [0, 6, 10, 14],
 [0, 6, 10, 15],
 [0, 6, 11, 12],
 [0, 6, 11, 13],
 [0, 6, 11, 14],
 [0, 6, 11, 15],
 [0, 7, 8, 12],
 [0, 7, 8, 13],
 [0, 7, 8, 14],
 [0, 7, 8, 15],
 [0, 7, 9, 12],
 [0, 7, 9, 13],
 [0, 7, 9, 14],
 [0, 7, 9, 15],
 [0, 7, 10, 12],
 [0, 7, 10, 13],
 [0, 7, 10, 14],
 [0, 7, 10, 15],
 [0, 7, 11, 

In [50]:
#bases = [[0,1,2,3],[4,5,6,7],[8,9,10,11],[12,13,14,15]]
from copy import deepcopy
total_options = []
for permut in tqdm(init_permut):
    first_pick = permut
    this_use = [[0,1,2,3],[4,5,6,7],[8,9,10,11],[12,13,14,15]]
    for i in range(4):
        this_use[i].remove(first_pick[i])
    options_here = permutations(*this_use)
    #print(options_here)
    for option in options_here:
        initial_case = deepcopy(this_use[:])
        for i in range(4):
            initial_case[i].remove(option[i])
        options_now = permutations(*initial_case)
        #print(options_now)
        for third_pick in options_now:
            third_case = deepcopy(initial_case)
            for i in range(4):
                third_case[i].remove(third_pick[i])
            options_third = permutations(*third_case)
            total_options.append([permut, option, third_pick, options_third[0]])
        

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:06<00:00, 41.78it/s]


In [59]:
animal_to_units = {}
for animal in df.animal:
    animal_to_units[animal] = df[df.animal==animal].unit_count.iloc[0]

In [60]:
animal_to_units

{'caf69': np.int64(366),
 'caf71': np.int64(248),
 'caf73': np.int64(389),
 'caf74': np.int64(289),
 'caf75': np.int64(309),
 'caf77': np.int64(112),
 'caf81': np.int64(423),
 'caf82': np.int64(210),
 'zbr106': np.int64(242),
 'zbr113': np.int64(38),
 'zbr114': np.int64(189),
 'zbr117': np.int64(241),
 'zbr118': np.int64(259),
 'zbr119': np.int64(229),
 'bth90': np.int64(279),
 'bth95': np.int64(301),
 'bth100': np.int64(193)}

In [93]:
def calculate_balance_for_group(grouping, details=False, tack_on=38):
    vals = []
    for group in grouping:
        vals.append(np.average([animal_to_units[animal_ordered[group[i]]] for i in range(len(group))]))
    
    if details: print(vals)
    min_ind = vals.index(min(vals))
    
    #Add in zbr113 to smallest grouping as tacked on guy
    vals[min_ind] = (vals[min_ind]*4 + tack_on)/5
    total_avg = np.average(vals)
    distance = sum([abs(vals[i]-total_avg)**2 for i in range(4)])
    return distance, min_ind

In [94]:
results = {'grouping':[], 'balance':[], 'tack_on_ind':[], 'to_animal':[]}
for layout in tqdm(total_options):
    bal, min_ind = calculate_balance_for_group(layout)
    results['grouping'].append(layout)
    results['balance'].append(bal)
    results['tack_on_ind'].append(min_ind)
    animal_layout = []
    for group in layout:
        animal_layout.append([animal_ordered[group[i]] for i in range(len(group))])
    animal_layout[min_ind].append('zbr113')
    results['to_animal'].append(animal_layout)
results = pd.DataFrame(results)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 331776/331776 [00:11<00:00, 28183.88it/s]


In [95]:
results

,grouping,balance,tack_on_ind,to_animal
0,"[[0, 4, 8, 12], [1, 5, 9, 13], [2, 6, 10, 14],...",7486.242500,3,"[[zbr106, zbr114, caf69, caf71], [bth90, zbr11..."
1,"[[0, 4, 8, 12], [1, 5, 9, 13], [2, 6, 10, 15],...",8625.442500,3,"[[zbr106, zbr114, caf69, caf71], [bth90, zbr11..."
2,"[[0, 4, 8, 12], [1, 5, 9, 13], [2, 6, 11, 14],...",3112.692500,1,"[[zbr106, zbr114, caf69, caf71], [bth90, zbr11..."
3,"[[0, 4, 8, 12], [1, 5, 9, 13], [2, 6, 11, 15],...",2925.192500,1,"[[zbr106, zbr114, caf69, caf71], [bth90, zbr11..."
4,"[[0, 4, 8, 12], [1, 5, 9, 13], [2, 7, 10, 14],...",5930.255000,3,"[[zbr106, zbr114, caf69, caf71], [bth90, zbr11..."
...,...,...,...,...
331771,"[[3, 7, 11, 15], [2, 6, 10, 14], [1, 4, 9, 13]...",7980.242500,0,"[[bth100, zbr119, caf82, caf75, zbr113], [bth9..."
331772,"[[3, 7, 11, 15], [2, 6, 10, 14], [1, 5, 8, 12]...",8863.421875,3,"[[bth100, zbr119, caf82, caf75], [bth95, zbr11..."
331773,"[[3, 7, 11, 15], [2, 6, 10, 14], [1, 5, 8, 13]...",16300.555000,3,"[[bth100, zbr119, caf82, caf75], [bth95, zbr11..."
331774,"[[3, 7, 11, 15], [2, 6, 10, 14], [1, 5, 9, 12]...",11134.716875,2,"[[bth100, zbr119, caf82, caf75], [bth95, zbr11..."


In [126]:
def animal_layout_to_descriptive(layout):
    totals = []
    for row in layout:
        totals.append(sum([animal_to_units[row[i]] for i in range(len(row))]))
    return totals
def check_layout_for_legal(layout):
    for row in layout:
        wy, wo, ay, ao = 0,0,0,0
        for animal in row:
            if (df[df.animal==animal].geno.iloc[0]=='wt') and (df[df.animal==animal].age_group.iloc[0]=='young'):
                wy+=1
            elif (df[df.animal==animal].geno.iloc[0]=='app_ps1' and df[df.animal==animal].age_group.iloc[0]=='old'):
                ao+=1
            elif df[df.animal==animal].geno.iloc[0]=='wt' and df[df.animal==animal].age_group.iloc[0]=='old':
                wo+=1
            else: 
                if animal != "zbr113":
                    ay += 1
        assert wy==wo==ay==ao

In [100]:
results[results.balance==min(results.balance)].grouping.iloc[0]

[[0, 4, 8, 14], [3, 7, 10, 12], [2, 5, 9, 13], [1, 6, 11, 15]]

In [120]:
best_layout = results[results.balance==min(results.balance)].to_animal.iloc[0]
best_layout


[['zbr106', 'zbr114', 'caf69', 'caf74'],
 ['bth100', 'zbr119', 'caf81', 'caf71'],
 ['bth95', 'zbr117', 'caf77', 'caf73', 'zbr113'],
 ['bth90', 'zbr118', 'caf82', 'caf75']]

In [127]:
print(animal_layout_to_descriptive(best_layout))
check_layout_for_legal(best_layout)

[np.int64(1086), np.int64(1093), np.int64(1081), np.int64(1057)]


In [89]:
df

,animal,geno,age,age_group,sum_chunks,chunks,unit_count
0,caf69,wt,358,old,"[108444, 21244, 104313]","[[199, 59, 63, 65, 62, 78, 61, 126, 199, 83, 8...",366
1,caf71,app_ps1,492,old,"[60581, 1659, 35141]","[[122, 181, 205, 195, 179, 328, 198, 131, 166,...",248
2,caf73,app_ps1,381,old,"[152900, 8475, 50362]","[[113, 205, 794, 478, 635, 674, 307, 281, 212,...",389
3,caf74,app_ps1,419,old,"[97297, 1925, 60293]","[[399, 251, 126, 147, 314, 383, 813, 388, 250,...",289
4,caf75,app_ps1,389,old,"[67096, 3068, 57185]","[[815, 306, 162, 726, 1769, 277, 85, 252, 67, ...",309
5,caf77,wt,394,old,"[32964, 6729, 30492]","[[441, 463, 93, 649, 377, 101, 276, 178, 212, ...",112
6,caf81,wt,455,old,"[92061, 25413, 105843]","[[211, 150, 90, 171, 397, 75, 87, 270, 119, 15...",423
7,caf82,wt,455,old,"[81495, 14513, 66498]","[[1585, 635, 351, 144, 157, 777, 925, 864, 142...",210
8,zbr106,wt,101,young,"[55262, 12533, 94664]","[[119, 613, 284, 68, 177, 193, 1201, 88, 363, ...",242
9,zbr113,app_ps1,91,young,"[3653, 757, 21263]","[[60, 77, 105, 32, 208, 21, 18, 151, 159, 168,...",38


In [91]:
(242+189+366+248), (279+259+210+309), (301+241+112+389+38), (193+229+423+289)

(1045, 1057, 1081, 1134)